# BUSINESS SALES PERFORMANCE ANALYTICS

## Task 1 | Future Interns - Data Science & Analytics Internship (2026)

### Project Overview
This project focuses on analyzing real-world transactional data to evaluate business performance, identify key revenue drivers, and provide strategic recommendations. The goal is to transform raw sales data into actionable business intelligence that can help stakeholders optimize operations and drive growth.

### Dataset Description
The analysis is conducted using the Online Retail Dataset, which contains all transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based, non-store online retail business.

### Objectives & Methodology
The workflow for this project involves:

- Data Cleaning: Handling missing values (specifically in CustomerID), removing duplicates, and filtering out cancellations or data anomalies.
- Feature Engineering: Calculating total sales per line item and extracting time-based features (Month, Day, Hour) from the InvoiceDate.
- Exploratory Data Analysis (EDA): Visualizing trends and distributions to understand the business landscape.
- Reporting: Building a professional dashboard and drafting data-driven recommendations.

### Research Questions
To guide the analysis, this project seeks to answer the following key questions:

- Revenue Drivers: Which products generate the most revenue and have the highest sales volume?
- Temporal Trends: How do sales and revenue fluctuate over time? Are there specific months or days with significant peaks?
- Geographic Performance: Which countries/regions are the most profitable, and where is the strongest customer base located?
- Customer Insights: What is the average order value, and how does customer distribution impact total sales?
- Strategic Focus: Based on the data, where should the business focus its marketing or inventory efforts to accelerate growth?

## DATA PREPROCESSING

In [2]:
import pandas as pd
import numpy as np

In [3]:
# load the data 
 
data = pd.read_csv('data.csv', encoding='ISO-8859-1')

In [4]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [7]:
data.tail()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,12/9/2011 12:50,4.95,12680.0,France


The dataset provides details of retail operations by tracking specific product sales, pricing, and customers

In [5]:
data.shape

(541909, 8)

The dataset contains 541 909 row entries and 8 column attributes.

In [6]:
data.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


On average, each transaction row consists of 9.5 units at a price of £4.60 per item. We observe some significant anomalies including negative values (min -80,995 quantity and -£11,062 price) which will have to be properly investigated and corrected if needed when cleaning.

## DATA CLEANING

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


We see already that there are some missing values and that CustomerID should be a string instead as it is an identifier and therefore categorical data.

In [10]:
data.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Description contains 1454 missing entries and CustomerID 135 080 missing entries. We check to see the percentage of missing values.

In [13]:
null_percentage = (data.isnull().sum() / len(data)) * 100

print(null_percentage.round(2))

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64


Less than 1% of data is missing from the Description column so we can remove these entries from the dataset. However the percentage of missing CustomerID values is 24.93% which is much larger than our cut off of 5%, we will need to find a way to impute these values.

In [15]:
# dropping the rows with missing values in Description
data = data.dropna(subset=['Description'])
data.shape

(540455, 8)

Dropped 1454 rows

Since CustomerID is categorical data, we convert the datatype to a string and replace all the missing values with 'Unknown'.

In [16]:
data['CustomerID'] = data['CustomerID'].fillna('Unknown')

In [17]:
data.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [19]:
# check for duplicates

data.duplicated().sum()

np.int64(5268)

In [20]:
duplicates = data[data.duplicated()]
print(duplicates.head())

    InvoiceNo StockCode                        Description  Quantity  \
517    536409     21866        UNION JACK FLAG LUGGAGE TAG         1   
527    536409     22866      HAND WARMER SCOTTY DOG DESIGN         1   
537    536409     22900    SET 2 TEA TOWELS I LOVE LONDON          1   
539    536409     22111       SCOTTIE DOG HOT WATER BOTTLE         1   
555    536412     22327  ROUND SNACK BOXES SET OF 4 SKULLS         1   

         InvoiceDate  UnitPrice CustomerID         Country  
517  12/1/2010 11:45       1.25    17908.0  United Kingdom  
527  12/1/2010 11:45       2.10    17908.0  United Kingdom  
537  12/1/2010 11:45       2.95    17908.0  United Kingdom  
539  12/1/2010 11:45       4.95    17908.0  United Kingdom  
555  12/1/2010 11:49       2.95    17920.0  United Kingdom  


There are 5268 exact duplicate rows. We need to remove these to avoid double counting sales.

In [22]:
# keeps the first occurrence
data = data.drop_duplicates()
data.duplicated().sum()

np.int64(0)

Now that the duplicates have been removed we check for outliers.

In [24]:
kurtosis_values = data.select_dtypes(include=['number']).kurt()

print("Kurtosis values:")
print(kurtosis_values)

Kurtosis values:
Quantity     119121.326173
UnitPrice     58275.430805
dtype: float64


Kurtosis is unnaturally high for the 'Quantity' and 'UnitPrice' columns, which may indicate the presence of outliers. We should investigate these columns further to understand the distribution of values and identify any potential outliers that could be affecting our analysis.

In [27]:
# use IQR to identify outliers

Q1_qty = data['Quantity'].quantile(0.25)
Q3_qty = data['Quantity'].quantile(0.75)
IQR_qty = Q3_qty - Q1_qty
lower_qty = Q1_qty - 1.5 * IQR_qty
upper_qty = Q3_qty + 1.5 * IQR_qty

outliers = data[(data['Quantity'] < lower_qty) | (data['Quantity'] > upper_qty)]
print(outliers)

       InvoiceNo StockCode                         Description  Quantity  \
9         536367     84879       ASSORTED COLOUR BIRD ORNAMENT        32   
26        536370     22728           ALARM CLOCK BAKELIKE PINK        24   
27        536370     22727           ALARM CLOCK BAKELIKE RED         24   
30        536370     21883                    STARS GIFT TAPE         24   
31        536370     10002         INFLATABLE POLITICAL GLOBE         48   
...          ...       ...                                 ...       ...   
541876    581585     84945  MULTI COLOUR SILVER T-LIGHT HOLDER        24   
541882    581585     21916     SET 12 RETRO WHITE CHALK STICKS        24   
541883    581585     84692         BOX OF 24 COCKTAIL PARASOLS        25   
541891    581586     23275    SET OF 3 HANGING OWLS OLLIE BEAK        24   
541892    581586     21217       RED RETROSPOT ROUND CAKE TINS        24   

            InvoiceDate  UnitPrice CustomerID         Country  
9        12/1/2010 8:34

In [28]:
Q1_price = data['UnitPrice'].quantile(0.25)
Q3_price = data['UnitPrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_price = Q1_price - 1.5 * IQR_price
upper_price = Q3_price + 1.5 * IQR_price

outliers_price = data[(data['UnitPrice'] < lower_price) | (data['UnitPrice'] > upper_price)]
print(outliers_price)

       InvoiceNo StockCode                      Description  Quantity  \
16        536367     22622   BOX OF VINTAGE ALPHABET BLOCKS         2   
45        536370      POST                          POSTAGE         3   
65        536374     21258       VICTORIAN SEWING BOX LARGE        32   
141      C536379         D                         Discount        -1   
151       536382     22839  3 TIER CAKE TIN GREEN AND CREAM         2   
...          ...       ...                              ...       ...   
541768    581578      POST                          POSTAGE         3   
541786    581578     22622   BOX OF VINTAGE ALPHABET BLOCKS         6   
541831    581579     22941     CHRISTMAS LIGHTS 10 REINDEER         4   
541849    581580     22894    TABLECLOTH RED APPLES DESIGN          2   
541892    581586     21217    RED RETROSPOT ROUND CAKE TINS        24   

            InvoiceDate  UnitPrice CustomerID         Country  
16       12/1/2010 8:34       9.95    13047.0  United Kingd

While IQR is an aggressive filtering method, I will use it here to remove extreme anomalies and bulk-order outliers that would otherwise skew the averages. This ensures that the subsequent visualizations reflect the behavior of the typical retail customer rather than system errors or rare wholesale transactions.

In [29]:
data = data[
    (data['Quantity'] >= lower_qty) & (data['Quantity'] <= upper_qty) &
    (data['UnitPrice'] >= lower_price) & (data['UnitPrice'] <= upper_price)
]
print(data.shape)

(438333, 8)


In [31]:
# standardise oolumn names 
data.columns = ['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']

data['customer_id'] = data['customer_id'].apply(lambda x: str(int(float(x))) if x != 'Unknown' and pd.notnull(x) else x)

data.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


In [32]:
data.to_csv('data_cleaned.csv', index=False)

## DATA ANALYSIS AND EXPLORATION